# **Task 5 — MAE**: where does a masked autoencoder keep what it learns?

This week's seminar was about [MAE](https://openaccess.thecvf.com/content/CVPR2022/html/He_Masked_Autoencoders_Are_Scalable_Vision_Learners_CVPR_2022_paper.html). It hides most patches of an image and trains an encoder and a small decoder to rebuild the missing pixels. Afterwards the decoder is thrown away and the encoder's features are used for recognition.

This notebook gives you CIFAR-10, the tiny ViT from Task 4, a **working MAE loop** and a **fixed evaluation**. The task is on the card.

<details>
<summary>Reach out here if you have any problems</summary>
Taras Kutsyk — taras.kutsyk@doctoral.uj.edu.pl
</details>

# Setup

**Step 0.** Runtime → Change runtime type → **T4 GPU**. Everything we need is already installed on Colab.

**Step 1. Data** (about 15 seconds). CIFAR-10: 50,000 training and 10,000 test images, 32×32, 10 classes, all kept on the GPU. **The labels are used only inside `knn_eval()` and `val_knn()`.**

In [ ]:
import io, time
import numpy as np, pandas as pd, requests, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Switch to a GPU runtime: Runtime -> Change runtime type -> T4 GPU"

def load(split):
    # returns: images (N, 3, 32, 32) as floats in [0, 1], labels (N), both on the GPU
    url = f"https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text/{split}-00000-of-00001.parquet"
    response = requests.get(url, timeout=120); response.raise_for_status()
    table = pd.read_parquet(io.BytesIO(response.content))
    images = np.stack([np.asarray(Image.open(io.BytesIO(row["bytes"])).convert("RGB")) for row in table["img"]])
    return torch.from_numpy(images).permute(0, 3, 1, 2).float().div(255).to(device), torch.tensor(table["label"].values).to(device)

train_images, train_labels = load("train")
test_images, test_labels = load("test")

MEAN = torch.tensor([.4914, .4822, .4465], device=device)[None, :, None, None]
STD = torch.tensor([.2470, .2435, .2616], device=device)[None, :, None, None]
normalize = lambda x: (x - MEAN) / STD   # apply to every image before it enters the network

print("train", train_images.shape, "test", test_images.shape)

**Step 2. The encoder** (DON'T modify it). The same tiny ViT as in Task 4: 4×4 patches, so 8×8 = 64 patches + the CLS token; 4 blocks, 128 dimensions, 4 heads, 0.8M parameters.

One addition for MAE: `encode(x, keep, block)` can run the network on **only some of the patches** (`keep` = their indices, as MAE's encoder does) and can **stop after any block** (`block` = 1..4). The feature used for k-NN is **the average of the patch tokens** (MAE's loss never trains the CLS token).

In [ ]:
class Block(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.qkv, self.proj = nn.Linear(dim, 3 * dim), nn.Linear(dim, dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(self.norm1(x)).reshape(B, T, 3, self.heads, D // self.heads).permute(2, 0, 3, 1, 4)
        x = x + self.proj(F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(B, T, D))
        return x + self.mlp(self.norm2(x))

class TinyViT(nn.Module):
    def __init__(self, dim=128, depth=4, heads=4, patch=4):
        super().__init__()
        self.dim = dim
        self.patch_embed = nn.Conv2d(3, dim, patch, patch)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.randn(1, dim, 8, 8) * .02)
        self.cls_pos = nn.Parameter(torch.zeros(1, 1, dim))
        self.blocks = nn.ModuleList(Block(dim, heads) for _ in range(depth))
        self.norm = nn.LayerNorm(dim)

    def encode(self, x, keep=None, block=4):
        # x     - (batch, 3, 32, 32) normalized images
        # keep  - (batch, n) indices of the patches to keep (0..63), or None for all 64
        # block - stop after this block (1..4); the final LayerNorm is applied only after block 4
        # returns: (batch, 1 + n, 128) tokens, the CLS token first
        tokens = (self.patch_embed(x) + self.pos).flatten(2).transpose(1, 2)
        if keep is not None:
            tokens = tokens.gather(1, keep[..., None].expand(-1, -1, self.dim))
        tokens = torch.cat([self.cls.expand(len(tokens), -1, -1) + self.cls_pos, tokens], 1)
        for b in self.blocks[:block]:
            tokens = b(tokens)
        return self.norm(tokens) if block == len(self.blocks) else tokens

    def forward(self, x, block=4):
        # returns: (batch, 128) the feature k-NN uses: the average of the patch tokens after `block`
        return self.encode(x, block=block)[:, 1:].mean(1)

print("parameters:", sum(p.numel() for p in TinyViT().parameters()))

**Step 3. The fixed evaluation** (DON'T modify it). `knn_eval(encoder, block)` passes the images through your frozen encoder **up to `block`**, stores the training features and labels, and classifies each test image by a weighted vote of its 20 nearest neighbours (a few seconds). It is the same k-NN as in Task 4.

- **`val_knn(encoder, block)`** does the same on a validation split: its queries are the last 5,000 training images, and its stored neighbours are the other 45,000. **Choose your settings with `val_knn()`, and use `knn_eval()` only for your final numbers.**

In [ ]:
@torch.no_grad()
def features(encoder, images, block):
    # returns: (N, 128) L2-normalized features of the frozen encoder after `block`
    with torch.autocast("cuda", dtype=torch.float16):
        return F.normalize(torch.cat([encoder(normalize(images[i:i + 1024]), block).float() for i in range(0, len(images), 1024)]), dim=-1)

def weighted_knn(bank, bank_labels, queries, query_labels, k=20, temperature=0.07):
    # returns: top-1 accuracy of a similarity-weighted vote of the k nearest stored features
    correct = 0
    for i in range(0, len(queries), 1024):
        similarities, neighbours = (queries[i:i + 1024] @ bank.T).topk(k, dim=1)
        votes = torch.zeros(len(similarities), 10, device=device).scatter_add_(1, bank_labels[neighbours], (similarities / temperature).exp())
        correct += (votes.argmax(1) == query_labels[i:i + 1024]).sum().item()
    return correct / len(queries)

def knn_eval(encoder, block=4):
    # The fixed evaluation for task 5. Do not change it; report exactly what it returns.
    # encoder - a TinyViT; block - which encoder block's features to use (1..4)
    # returns: top-1 accuracy on the CIFAR-10 test set
    was_training = encoder.training
    encoder.eval()
    accuracy = weighted_knn(features(encoder, train_images, block), train_labels, features(encoder, test_images, block), test_labels)
    encoder.train(was_training)
    return accuracy

def val_knn(encoder, block=4):
    # The same as knn_eval, on the validation split. Use this one for every choice you tune.
    # returns: top-1 accuracy on the last 5,000 training images, with the other 45,000 as neighbours
    was_training = encoder.training
    encoder.eval()
    all_features = features(encoder, train_images, block)
    accuracy = weighted_knn(all_features[:45000], train_labels[:45000], all_features[45000:], train_labels[45000:])
    encoder.train(was_training)
    return accuracy

**Step 4. Baselines.** You should get about **45.7%** on raw pixels and about **32%** with the untrained encoder, at every block.

In [ ]:
pixels = lambda x: F.normalize(normalize(x).flatten(1), dim=-1)
print("raw pixels, k-NN accuracy:", weighted_knn(pixels(train_images), train_labels, pixels(test_images), test_labels))
torch.manual_seed(0)
untrained = TinyViT().to(device)
print("untrained encoder, k-NN accuracy per block:", [round(knn_eval(untrained, b), 4) for b in range(1, 5)])

# Part 1: the pretrained models

**Step 5.** Two ViT-B/16 models of the same size (12 blocks, 768 dimensions), pretrained on ImageNet, from Hugging Face. They are used as they are: no training in Part 1.

- **`mae`**: the paper's own pretrained MAE, encoder `mae.vit` (12 blocks) and decoder `mae.decoder` (8 blocks, 512 dimensions) ([`facebook/vit-mae-base`](https://huggingface.co/facebook/vit-mae-base), [docs](https://huggingface.co/docs/transformers/model_doc/vit_mae)). By default it hides 75% of the patches even at test time; we set `mask_ratio=0` so it sees the whole image (it still shuffles the patch order internally, and with nothing hidden its reconstruction loss is NaN: ignore it).
- **`dino`**: a ViT-B/16 trained with DINO ([`facebook/dino-vitb16`](https://huggingface.co/facebook/dino-vitb16), [docs](https://huggingface.co/docs/transformers/model_doc/vit)).

Both expect 224×224 images normalized with ImageNet statistics: `to_224(images)` does that for CIFAR images. Use `weighted_knn()` from Step 3 if you need a k-NN classifier. A forward pass of all 60,000 images through one model takes about 2.5 minutes on a T4.

In [ ]:
from transformers import ViTMAEForPreTraining, ViTModel

mae = ViTMAEForPreTraining.from_pretrained("facebook/vit-mae-base", mask_ratio=0.0).to(device).half().eval()   # encoder: mae.vit, decoder: mae.decoder
dino = ViTModel.from_pretrained("facebook/dino-vitb16", add_pooling_layer=False).to(device).half().eval()

IMAGENET_MEAN = torch.tensor([.485, .456, .406], device=device)[None, :, None, None]
IMAGENET_STD = torch.tensor([.229, .224, .225], device=device)[None, :, None, None]

def to_224(images):
    # CIFAR images (batch, 3, 32, 32) in [0, 1] -> (batch, 3, 224, 224) in fp16, ImageNet-normalized, as both models expect
    images = F.interpolate(images, size=224, mode="bicubic", align_corners=False).clamp(0, 1)
    return ((images - IMAGENET_MEAN) / IMAGENET_STD).half()

with torch.no_grad():
    print("MAE encoder output:", mae.vit(pixel_values=to_224(test_images[:8])).last_hidden_state.shape)   # (8, 1 + 196 patches, 768)
    print("DINO output:", dino(pixel_values=to_224(test_images[:8])).last_hidden_state.shape)

# Part 2: your own small MAE

**Step 6.** MAE as in the paper (Fig. 1 and §3), at our scale:

```
image -> 64 patches -> shuffle, keep the first 25% (16 patches)
      -> ENCODER (TinyViT) on the 16 visible patches only
      -> project to the decoder width, add one learned "mask token" per hidden patch,
         un-shuffle back to image order, add positions
      -> DECODER (dec_depth blocks) -> 48 pixel values (4x4x3) per patch
      -> loss: mean squared error on the HIDDEN patches only
```

The target is each patch's pixels normalized by that patch's own mean and standard deviation (the paper's "pixel (w/ norm)", Table 1d). Augmentation is crop-and-resize plus flip only, as in the paper. `train_mae` returns the trained MAE and the loss per epoch; `show_reconstructions` shows hidden / rebuilt / original images.

For Part 2, copy this cell and change it: everything here is yours to modify, except the encoder, the 20-epoch budget and the rule that the only training signal is rebuilding the hidden patches.

In [ ]:
def augment(images):
    # crop-and-resize (35-100% of the area) and flip; images - (batch, 3, 32, 32) in [0, 1]; returns: the same shape, normalized
    n = len(images)
    scale = (.35 + .65 * torch.rand(n, device=device)).sqrt()
    shift_x, shift_y = (1 - scale) * (2 * torch.rand(n, device=device) - 1), (1 - scale) * (2 * torch.rand(n, device=device) - 1)
    flip = torch.where(torch.rand(n, device=device) < .5, -1., 1.)
    theta = torch.zeros(n, 2, 3, device=device)
    theta[:, 0, 0], theta[:, 1, 1], theta[:, 0, 2], theta[:, 1, 2] = scale * flip, scale, shift_x, shift_y
    grid = F.affine_grid(theta, images.shape, align_corners=False)
    return normalize(F.grid_sample(images, grid, align_corners=False, padding_mode="reflection"))

def patchify(x):
    # (batch, 3, 32, 32) -> (batch, 64, 48): row-major 8x8 grid of 4x4x3 patches
    return x.unfold(2, 4, 4).unfold(3, 4, 4).permute(0, 2, 3, 1, 4, 5).reshape(len(x), 64, 48)

def unpatchify(p):
    # (batch, 64, 48) -> (batch, 3, 32, 32), the inverse of patchify
    return p.reshape(len(p), 8, 8, 3, 4, 4).permute(0, 3, 1, 4, 2, 5).reshape(len(p), 3, 32, 32)

class MAE(nn.Module):
    def __init__(self, dec_depth=1, dec_dim=64, ratio=.75):
        super().__init__()
        self.encoder, self.ratio = TinyViT(), ratio
        self.to_decoder = nn.Linear(self.encoder.dim, dec_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, dec_dim))
        self.dec_pos = nn.Parameter(torch.randn(1, 65, dec_dim) * .02)
        self.dec_blocks = nn.ModuleList(Block(dec_dim, 4) for _ in range(dec_depth))
        self.dec_norm, self.to_pixels = nn.LayerNorm(dec_dim), nn.Linear(dec_dim, 48)

    def forward(self, x):
        # x - (batch, 3, 32, 32) normalized images
        # returns: loss, pred (batch, 64, 48) in normalized-patch units, hidden (batch, 64): 1 = hidden patch
        B, n_keep = len(x), round(64 * (1 - self.ratio))
        shuffle = torch.rand(B, 64, device=x.device).argsort(1)       # a random order of the 64 patches per image
        unshuffle, keep = shuffle.argsort(1), shuffle[:, :n_keep]       # keep the first n_keep of that order
        latent = self.to_decoder(self.encoder.encode(x, keep))          # (B, 1 + n_keep, dec_dim)
        tokens = torch.cat([latent[:, 1:], self.mask_token.expand(B, 64 - n_keep, -1)], 1)
        tokens = tokens.gather(1, unshuffle[..., None].expand(-1, -1, tokens.shape[-1]))  # back to image order
        tokens = torch.cat([latent[:, :1], tokens], 1) + self.dec_pos
        for b in self.dec_blocks:
            tokens = b(tokens)
        pred = self.to_pixels(self.dec_norm(tokens))[:, 1:].float()
        target = patchify(x)
        target = (target - target.mean(-1, keepdim=True)) / (target.var(-1, keepdim=True) + 1e-6).sqrt()
        hidden = torch.ones(B, 64, device=x.device).scatter_(1, keep, 0)
        loss = (((pred - target) ** 2).mean(-1) * hidden).sum() / hidden.sum()
        return loss, pred, hidden

def train_mae(dec_depth=1, dec_dim=64, ratio=.75, epochs=20, seed=0, batch_size=512, lr=1.5e-3):
    # returns: {"mae": the trained MAE (its encoder is mae.encoder), "history": [{"epoch", "loss"}, ...]}
    assert epochs <= 20, "the budget is 20 epochs per run"
    torch.manual_seed(seed)
    mae = MAE(dec_depth, dec_dim, ratio).to(device)
    optimizer = torch.optim.AdamW(mae.parameters(), lr=lr, betas=(.9, .95), weight_decay=.05)
    steps = epochs * (len(train_images) // batch_size)
    schedule = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: min(1, (s + 1) / 200) * .5 * (1 + np.cos(np.pi * s / steps)))
    scaler = torch.amp.GradScaler("cuda")
    history, start = [], time.time()
    for epoch in range(1, epochs + 1):
        order, total = torch.randperm(len(train_images), device=device), 0.
        for i in range(len(order) // batch_size):
            with torch.autocast("cuda", dtype=torch.float16):
                loss, _, _ = mae(augment(train_images[order[i * batch_size:(i + 1) * batch_size]]))
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update(); schedule.step()
            total += loss.item()
        history.append(dict(epoch=epoch, loss=total / (len(order) // batch_size)))
        val = f", val_knn {val_knn(mae.encoder):.4f}" if epoch % 5 == 0 else ""
        print(f"epoch {epoch:2d} ({time.time() - start:4.0f}s): loss {history[-1]['loss']:.4f}{val}", flush=True)
    return dict(mae=mae, history=history)

@torch.no_grad()
def show_reconstructions(mae, n=6, seed=0):
    # rows: what the encoder sees / the decoder's guess pasted into the hidden patches / the original
    torch.manual_seed(seed)
    x = test_images[:n]
    mae.eval()
    _, pred, hidden = mae(normalize(x))
    mae.train()
    # The decoder predicts each hidden patch normalized by its own mean and std, which it never sees. To display a guess,
    # we give it the average mean and std of the VISIBLE patches of the same image (using the hidden patch's own would leak the answer).
    patches, h = patchify(x), hidden[..., None]
    mean, std = patches.mean(-1, keepdim=True), (patches.var(-1, keepdim=True) + 1e-6).sqrt()
    visible_avg = lambda v: (v * (1 - h)).sum(1, keepdim=True) / (1 - h).sum(1, keepdim=True)
    guess = pred * visible_avg(std) + visible_avg(mean)
    rows = [unpatchify(patches * (1 - h) + .5 * h), unpatchify(patches * (1 - h) + guess * h), x]
    fig, axes = plt.subplots(3, n, figsize=(1.6 * n, 5))
    for r, (images, name) in enumerate(zip(rows, ["input", "rebuilt", "original"])):
        for j in range(n):
            axes[r, j].imshow(images[j].clamp(0, 1).permute(1, 2, 0).cpu()); axes[r, j].axis("off")
        axes[r, 0].set_title(name, loc="left", fontsize=9)
    plt.tight_layout(); plt.show()

**Step 7. Reference run.** MAE with a 1-block decoder, 20 epochs, masking 75%. It takes about 1 minute, and you should get about **46.5%** with `knn_eval()` after block 4 (other seeds: 46.8%, 47.1%).

In [ ]:
reference = train_mae()
encoder = reference["mae"].encoder
print("k-NN per encoder block:", [round(knn_eval(encoder, b), 4) for b in range(1, 5)])
print("peak GPU memory, GB:", round(torch.cuda.max_memory_allocated() / 2**30, 1))
show_reconstructions(reference["mae"])

# Your solution

**Part 1:** is the paper's explanation true for the real MAE? **Part 2:** your own MAE, as good as you can make it on `knn_eval()`. Results vanish with the runtime, so save what you need to Drive. Good luck!

In [ ]:
# Part 1

In [ ]:
# Part 2